# Aria — QLoRA fine-tune (free Colab T4)

Trains a small instruct model to emit **Aria's exact JSON contract**:

```json
{"sentences": [{"text": "...", "emotion": "...", "intensity": 2, "effect": "none", "style_prompt": "..."}],
 "tool_calls": [{"name": "open_app", "arguments": {"name": "YouTube"}}]}
```

**Runtime → Change runtime type → T4 GPU** before running.

Everything is validated against `training/aria/schema.py`, which mirrors the Android app's parser — so a passing score means the app can actually consume the output.

## 1. Setup

In [ ]:
!nvidia-smi | head -12
!pip -q install 'transformers>=4.45' 'trl>=0.12' peft accelerate bitsandbytes datasets 2>&1 | tail -3

In [ ]:
import os, torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
assert torch.cuda.is_available(), "Enable a GPU: Runtime > Change runtime type > T4"

# Get the code. Replace with your fork/branch if needed.
REPO = "https://github.com/sumitsanataniirajput/Llm-corpus.git"
BRANCH = "arena/d0077f57-llm-corpus"

if not os.path.isdir("/content/Llm-corpus"):
    !git clone -b {BRANCH} --depth 1 {REPO} /content/Llm-corpus
%cd /content/Llm-corpus/training/aria

## 2. (Re)generate the dataset

Deterministic — same seed, same file. Skip to reuse the committed data.

In [ ]:
!python generate_dataset.py --total 6000 --out data/
!wc -l data/*.jsonl

## 3. Sanity-check the harness

The oracle replays the reference answers, so it must score **exactly 100**. If it doesn't, the harness is broken — not the model.

In [ ]:
!python evaluate.py --backend oracle --limit 300

## 4. Baseline: how good is the *untrained* model?

Record this before training. Without it you cannot tell whether fine-tuning helped.

In [ ]:
BASE = "Qwen/Qwen2.5-1.5B-Instruct"   # 3B if you have headroom
!python evaluate.py --backend hf --model {BASE} --limit 200 --json-out results/baseline.json

## 5. Train

In [ ]:
!python train_qlora.py \
    --base {BASE} \
    --out /content/aria-lora \
    --epochs 2 \
    --batch-size 2 \
    --grad-accum 8

## 6. Evaluate the fine-tune

In [ ]:
!python evaluate.py --backend hf --model {BASE} --adapter /content/aria-lora \
    --limit 200 --json-out results/finetuned.json

## 7. Baseline vs fine-tuned side by side

In [ ]:
import json
b = json.load(open("results/baseline.json"))
f = json.load(open("results/finetuned.json"))

print(f"{'metric':18s} {'baseline':>10s} {'fine-tuned':>12s} {'delta':>8s}")
print("-" * 52)
for k in ["json_rate", "schema_rate", "emotion_acc", "tool_em", "score"]:
    d = f[k] - b[k]
    flag = "  <-- better" if d > 0.5 else ("  <-- WORSE" if d < -0.5 else "")
    print(f"{k:18s} {b[k]:10.2f} {f[k]:12.2f} {d:+8.2f}{flag}")

assert f["schema_rate"] >= b["schema_rate"], "fine-tune regressed on schema conformance"
print("\nSchema conformance held. Copy the winning run into the app config.")

## 8. Spot-check real conversations

Numbers can hide embarrassing output. Read a few.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch, json
from prompt import system_prompt

tok = AutoTokenizer.from_pretrained(BASE)
m = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype="auto", device_map="auto")
m = PeftModel.from_pretrained(m, "/content/aria-lora").eval()

prompts = [
    "open youtube",
    "set an alarm for 6:30",
    "my dog passed away",
    "someone broke my phone",
    "call mom",
    "scroll down",
]
for p in prompts:
    msgs = [{"role":"system","content":system_prompt()},{"role":"user","content":p}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = tok(text, return_tensors="pt").to(m.device)
    with torch.no_grad():
        out = m.generate(**inp, max_new_tokens=400, do_sample=False,
                         pad_token_id=tok.eos_token_id)
    print("=" * 60)
    print("USER:", p)
    print(tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)[:500])

## 9. Save the adapter

The LoRA adapter is ~30–80 MB — small enough to download from Colab.

In [ ]:
!zip -qr /content/aria-lora.zip /content/aria-lora
from google.colab import files
files.download("/content/aria-lora.zip")

## Notes

- **This does not change the Android app.** Aria currently calls hosted APIs (Gemini/GPT/Claude) with your key. Serving a local model means either bundling a small GGUF on-device (MediaPipe/llama.cpp) or hosting it behind an OpenAI-compatible endpoint and pointing Aria's `OPENAI_COMPATIBLE` provider at it.
- A 1.5B model will be **less knowledgeable** than a frontier API. The realistic win is *format reliability* — never malformed JSON, never a missed tool call.
- If `score` doesn't beat the baseline, don't ship it. That is the entire point of step 7.